# Train a face attack decision model

Fine-tunes `unsloth/Qwen3.5-4B` with Unsloth's `FastDecisionModel` so that, for one cropped face image, it answers:

* `is_bona_fide` (yes/no): is this a live, unaltered face?
* `attack_type` (choice): bona fide, print, replay, mask, face swap, face reenactment or fully synthetic.

Then it scores the model on subjects it never saw, with ISO/IEC 30107-3 metrics broken down by attack type and by generator.

**Runtime:** Colab Pro with an L4 or A100 GPU. On a T4, Qwen3.5 trains in float32 and is very slow.

**Smoke mode** (`SMOKE = True`, the default) needs no data: it generates noise images to check that the pipeline runs. Only the repository has to be reachable through `REPO_SOURCE`.

**Before a real run:** build the private dataset with `build_dataset.ipynb`, add a Colab secret `HF_TOKEN` with write access, and set `SMOKE = False`. The adapter and its report are uploaded to a private Hugging Face model repository.

**Licenses:** the public face datasets are research-only, and Unsloth's decision model code is AGPL-3.0. Keep the trained adapter for research.

### Configuration

In [ ]:
# SMOKE = True runs the whole pipeline on generated noise images with arbitrary labels. It checks
# that installation, loading, training, calibration and prediction work on this GPU; its metrics
# mean nothing. SMOKE = False trains on the private dataset built by build_dataset.ipynb and scores
# it under the shared protocol (face_decision_lab.protocol), like clip_baseline.ipynb.
SMOKE = True

# Where pip installs this repository from: a copy in Drive or a git URL ("git+https://...").
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"   # private; needs the HF_TOKEN Colab secret
HF_MODEL = "paulohenriquevn/face-decision-qwen35-4b"  # private repo the adapter is uploaded to

if SMOKE:
    DATA_DIR = "/content/smoke_data"
    OUTPUT_DIR = "/content/runs/smoke"  # throwaway
    MAX_STEPS = 5
    MAX_TRAIN_SAMPLES = None
    MAX_CALIBRATION_SAMPLES = None
else:
    OUTPUT_DIR = "/content/runs/qwen35_4b"
    MAX_STEPS = 60                  # short run; set to -1 for a full epoch
    # A short run sees MAX_STEPS x 32 rows, so building items for the rest would only cost time.
    MAX_TRAIN_SAMPLES = 60 * 32
    MAX_CALIBRATION_SAMPLES = 1000

### Installation

In [ ]:
if not SMOKE:
    from google.colab import userdata
    from huggingface_hub import login

    login(token = userdata.get("HF_TOKEN"))

In [ ]:
%%capture
import importlib.util
import os

!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install -qqq "{REPO_SOURCE}"

### Data

Splits come from `face_decision_lab.splits`, not from `FastDecisionModel.split_holdout`: that one shuffles rows, and since every frame is a row, frames of one video would land on both sides.

In [ ]:
from collections import Counter

from face_decision_lab.protocol import SEED
from face_decision_lab.rows import decision_row, local_image
from face_decision_lab.splits import capped

if SMOKE:
    from face_decision_lab.manifest import load_manifest
    from face_decision_lab.smoke import SMOKE_TEST_SOURCE, SMOKE_TRAIN_SOURCE, write_smoke_dataset
    from face_decision_lab.splits import cross_dataset_split, group_split

    write_smoke_dataset(DATA_DIR, seed = SEED)
    samples = load_manifest(f"{DATA_DIR}/manifest.csv")
    images = {s.image_path: local_image(s, DATA_DIR) for s in samples}
    train_samples, test_samples = cross_dataset_split(samples, {SMOKE_TRAIN_SOURCE}, {SMOKE_TEST_SOURCE})
    train_samples, calibration_samples = group_split(train_samples, 0.5, seed = SEED)
else:
    from face_decision_lab.hub import load_hub_samples
    from face_decision_lab.protocol import protocol_split

    samples, hub_images = load_hub_samples(HF_DATASET)
    images = {s.image_path: image for s, image in zip(samples, hub_images, strict = True)}
    train_samples, calibration_samples, test_samples = protocol_split(samples)
train_samples = capped(train_samples, MAX_TRAIN_SAMPLES, SEED)
calibration_samples = capped(calibration_samples, MAX_CALIBRATION_SAMPLES, SEED)

for name, part in [("train", train_samples), ("calibration", calibration_samples), ("test", test_samples)]:
    print(name, len(part), dict(Counter(s.attack_type for s in part)))

### Model

In [ ]:
from unsloth import DecisionTrainer, FastDecisionModel, is_bfloat16_supported

model, processor = FastDecisionModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048,
    load_in_4bit = True,
)
# LoRA trains the language layers; Unsloth keeps the vision tower frozen.
model = FastDecisionModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
train_items, train_report = FastDecisionModel.build_dataset(
    [decision_row(s, images[s.image_path]) for s in train_samples], processor, model
)
calibration_items, calibration_report = FastDecisionModel.build_dataset(
    [decision_row(s, images[s.image_path]) for s in calibration_samples], processor, model
)
print("train", train_report)
print("calibration", calibration_report)

### Train

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = processor,
    train_dataset = train_items,
    eval_dataset = calibration_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = f"{OUTPUT_DIR}/checkpoints",
        report_to = "none",
        seed = SEED,
    ),
)
trainer_stats = trainer.train()

### Calibrate on held-out training subjects

In [ ]:
FastDecisionModel.calibrate(model, processor, calibration_items)

### Evaluate on held-out subjects

`attack_score` is `1 - P(is_bona_fide = true)`. Metrics follow ISO/IEC 30107-3: APCER is reported for the worst attack type.

In [ ]:
import json, time

from face_decision_lab.labels import QUESTIONS, STATE
from face_decision_lab.protocol import evaluation_report
from face_decision_lab.rows import attack_score, open_image

FastDecisionModel.for_inference(model)
scores, type_hits, latencies = [], 0, []
for sample in test_samples:
    image = open_image(images[sample.image_path])
    started = time.perf_counter()
    answers = FastDecisionModel.predict(model, processor, STATE, QUESTIONS, images = [image])
    latencies.append(time.perf_counter() - started)
    scores.append(attack_score(answers))
    type_hits += answers["attack_type"]["answer"] == sample.attack_type

report = evaluation_report(test_samples, scores)
report["attack_type_accuracy"] = type_hits / len(test_samples)
report["predict_p95_ms"] = sorted(latencies)[int(0.95 * (len(latencies) - 1))] * 1000
print(json.dumps(report, indent = 2))

### Save

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")  # LoRA adapter, decision head and calibration
if not SMOKE:
    from pathlib import Path

    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(HF_MODEL, private = True, exist_ok = True)
    api.upload_folder(folder_path = f"{OUTPUT_DIR}/adapter", repo_id = HF_MODEL, commit_message = f"Adapter after {MAX_STEPS} steps")
    Path(f"{OUTPUT_DIR}/report.json").write_text(json.dumps(report, indent = 2))
    api.upload_file(path_or_fileobj = f"{OUTPUT_DIR}/report.json", path_in_repo = "report.json", repo_id = HF_MODEL)